In [19]:
# LCEL 문법을 사용하여 chain 을 생성
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatOpenAI 모델을 인스턴스화
model = ChatOpenAI()

# 주어진 토픽에 대한 농담을 요청하는 프롬프트 템플릿을 생성
prompt = PromptTemplate.from_template('{topic}에 대하여 3문장으로 설명해줘.')

# 프롬프트와 모델을 연결하여 대화 체인을 생성
chain = prompt | model | StrOutputParser()

In [20]:
# 주어진 토픽 리스트를 batch 처리하는 함수 호출
chain.batch([{'topic':'ChatGPT'}, {'topic':'Instagram'}])

['ChatGPT는 인공지능 챗봇 서비스로, 사람과 자연스럽게 대화를 나눌 수 있습니다. 맥락을 이해하고 학습하여 정확하고 유용한 답변을 제공합니다. 다양한 주제에 대한 질문에 응답하며, 상호작용을 통해 지식을 확장할 수 있습니다.',
 'Instagram은 사진과 동영상을 공유하고 다른 사람들과 소통할 수 있는 소셜 미디어 플랫폼이다. 사용자들은 팔로워들의 업데이트를 볼 수 있고 좋아요와 댓글을 남길 수 있다. 또한 스토리 기능을 통해 24시간 동안 임시로 컨텐츠를 올릴 수 있는 기능도 있다.']

In [21]:
# 다섯 개의 질문을 처리 -> 최대 3개의 작업을 동시에 처리하도록 설정
chain.batch(
    [
        {'topic':'ChatGPT'},
        {'topic':'Instagram'},
        {'topic':'멀티모달'},
        {'topic':'프로그래밍'},
        {'topic':'머신러닝'}
    ],
    config={'max_concurrency':3},
)

['ChatGPT는 대화형 AI 모델로, 자연어로 대화하며 다양한 주제에 대해 대화할 수 있습니다. 사용자의 질문에 답변하거나 특정 주제에 대한 정보를 제공해줄 뿐만 아니라, 일상적인 대화도 가능합니다. 또한 ChatGPT는 세련된 언어모델을 가지고 있어 자연스러운 응답을 생성합니다.',
 '인스타그램은 사진과 동영상을 공유하고 서로의 소식을 나눌 수 있는 소셜 미디어 플랫폼입니다. 다양한 필터와 편집 기능을 제공하여 사용자가 자신만의 콘텐츠를 만들 수 있습니다. 팔로워와 좋아요를 통해 사용자들 간의 상호 소통과 교류가 이루어집니다.',
 '멀티모달은 여러 가지 다른 형태의 매체를 결합하여 사용자에게 정보를 제공하거나 상호 작용하는 기술이다. 이는 텍스트, 음성, 이미지, 동영상 등 다양한 형식을 이용하여 사용자 경험을 향상시키고 효율적인 의사 소통을 도와준다. 현대의 디지털 기술과 인공지능 기술의 발전으로 멀티모달 기술이 더욱 발전하여 사용자에게 다양한 형태의 경험을 제공하게 되었다.',
 '프로그래밍은 컴퓨터에게 특정 작업을 수행하도록 지시하는 일련의 명령어를 작성하는 과정입니다. 이를 통해 사용자는 원하는 결과를 얻을 수 있고, 문제를 해결할 수 있습니다. 프로그래밍은 문제 해결 능력과 논리적 사고를 키워주며, 다양한 분야에서 활용됩니다.',
 '머신러닝은 데이터를 사용하여 패턴이나 규칙을 학습하는 인공지능 기술이다. 이를 통해 모델을 만들어 새로운 데이터에 대한 예측을 할 수 있으며, 스스로 학습하여 정확도를 높일 수 있다. 머신러닝은 다양한 분야에서 활용되며, 예측, 분류, 군집화 등 다양한 작업에 사용된다.']

In [22]:
# parallel 병렬성
from langchain_core.runnables import RunnableParallel

# 체인 1 : {country}의 수도를 물어보는 체인
chain1 = (
    PromptTemplate.from_template('{country}의 수도는 어디야?') | model | StrOutputParser()
)

# 체인 2 : {country}의 면적을 물어보는 체인
chain2 = (
    PromptTemplate.from_template('{country}의 면적은 얼마야?') | model | StrOutputParser()
)

# 2개의 체인을 동시에 생성하는 병렬 실행 체인을 생성
combined = RunnableParallel(capital=chain1, area=chain2)

In [23]:
# 체인 1 실행
chain1.invoke({'country':'대한민국'})

'대한민국의 수도는 서울이에요.'

In [24]:
# 체인 2 실행
chain2.invoke({'country':'미국'})

'미국의 총 면적은 9,834만 제곱킬로미터이다.'

In [25]:
# 병렬 체인을 실행
combined.invoke({'country':'대한민국'})

{'capital': '서울이 대한민국의 수도입니다.', 'area': '대한민국의 면적은 약 100,363㎢ 입니다.'}

In [26]:
chain1.batch([{'country':'대한민국'}, {'country':'미국'}])

['대한민국의 수도는 서울이다.', '미국의 수도는 워싱턴 D.C.입니다.']

In [27]:
chain2.batch([{'country':'대한민국'}, {'country':'미국'}])

['대한민국의 면적은 약 100,284km² 이다.', '미국의 총 면적은 약 9,826만 제곱 킬로미터입니다.']

In [28]:
# 배치에서의 병렬처리
combined.batch([{'country':'대한민국'}, {'country':'미국'}])

[{'capital': '대한민국의 수도는 서울이야.', 'area': '대한민국의 면적은 약 100,363.43km² 입니다.'},
 {'capital': '미국의 수도는 워싱턴 D.C.입니다.', 'area': '미국의 총 면적은 약 9,833,520 km² 입니다.'}]

In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template('{num}의 10는? ')
llm = ChatOpenAI(temperature=0)

chain = prompt | llm

In [30]:
# chain을 실행
chain.invoke({'num':5})

AIMessage(content='5의 10승은 9765625입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C32Au4Gu2LcIfJzpRgeseK5sfI7fN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--214528b6-88c4-42bf-977b-8bc8644f8a07-0', usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [31]:
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke({'num':10})

{'num': 10}

In [32]:
runnable_chain = {'num':RunnablePassthrough()} | prompt | ChatOpenAI()

runnable_chain.invoke(10)

AIMessage(content='10의 10승은 10,000,000,000이다. (10억)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 14, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C32AwlQYeuQXlUeGmpD1w6zs3ExEf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--dc91d107-a78d-45a3-892e-e13dd6321abb-0', usage_metadata={'input_tokens': 14, 'output_tokens': 23, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [33]:
RunnablePassthrough.assign(new_num=lambda x:x['num']*3).invoke({'num':1})
# new_num의 값을 딕셔너리에 추가

{'num': 1, 'new_num': 3}

In [34]:
from langchain_core.runnables import RunnableParallel
runnable = RunnableParallel(
    passed = RunnablePassthrough(), # 입력된 데이터를 그대로 통과시키는 역할
    extra = RunnablePassthrough.assign(mult=lambda x: x['num'] * 3),
    # 입력된 딕셔너리의 'num'키에 해당하는 값에 1을 더한다.
    modified = lambda x: x['num'] + 1,
)

runnable.invoke({'num':1})
# num : 1이 패스돼서 passed, extra, modified를 읽어들임

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

In [35]:
chain1 = (
    {'country':RunnablePassthrough()}
    | PromptTemplate.from_template('{country}의 수도는?')
    | ChatOpenAI()
)
chain2 = (
    {'country':RunnablePassthrough()}
    | PromptTemplate.from_template('{country}의 면적은?')
    | ChatOpenAI
)

In [ ]:
combined_chain = RunnableParallel(captial=chain1, area=chain2)
combined_chain.invoke('대한민국')

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given